# PrivaDE workflow

In this scenario, we assume Bob has multiple data points to contribute to Alice's ML model. Alice is trying to value the dataset as a whole, judging on the diversity, uncertainty of the datasets as well as the current model's performance on the dataset. Moreover, the parties are assumed to be malicious, which means they might deviate from the protocol to maximize their own utility.

## Part 0: Setup

We set up Alice's model and Bob's dataset.

In [1]:
import os
import torch
import sys
import random
sys.path.append('..')  # Add privade directory to path
from privade.data import get_dataset
from privade.models import get_model
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N = 1000 #Bob's dataset size

full_model = get_model('resnet20', 'cifar10')
full_data = get_dataset('cifar10')

# Randomly select 1000 images as Bob's dataset
indices = random.sample(range(len(full_data)), N)
bob_images = np.array([full_data[i][0].numpy() for i in indices])
bob_labels = np.array([full_data[i][1]  for i in indices])

# Ramdomly select 1000 images as Alice's initial dataset
indices = random.sample(range(len(full_data)), N)
alice_images = np.array([full_data[i][0].numpy() for i in indices])
alice_labels = np.array([full_data[i][1]  for i in indices])

# Train the model for a few epochs
alice_images_tensor = torch.FloatTensor(alice_images)
alice_labels_tensor = torch.LongTensor(alice_labels)
alice_dataset = TensorDataset(alice_images_tensor, alice_labels_tensor)
alice_dataloader = DataLoader(alice_dataset, batch_size=32, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(full_model.parameters(), lr=0.001)
full_model.train()
for epoch in range(5):
    running_loss = 0.0
    for images, labels in alice_dataloader:
        optimizer.zero_grad()
        outputs = full_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}/5, Loss: {running_loss/len(alice_dataloader):.4f}')
    
# Save models and datasets
torch.save(bob_images, 'data/bob_images.pth')
torch.save(bob_labels, 'data/bob_labels.pth')
torch.save(full_model.state_dict(), 'data/alice_full_model.pth')

Files already downloaded and verified
Epoch 1/5, Loss: 2.4561
Epoch 2/5, Loss: 1.9467
Epoch 3/5, Loss: 1.7436
Epoch 4/5, Loss: 1.5828
Epoch 5/5, Loss: 1.3664


### Model distillation

For Alice's preprocessing, she has an optional step to do model distillation, to obtain a smaller model for data evaluation.

In [2]:
from privade.distillation import train_distilled_model

student_model = get_model('lenet5', 'cifar10')

kd_train_loader = DataLoader(alice_dataset, batch_size=32, shuffle=True)

trained_student_model = train_distilled_model(full_model, student_model, kd_train_loader, kd_train_loader,epochs=10)

torch.save(trained_student_model.state_dict(), "data/alice_student_model.pth")

Starting knowledge distillation training on cuda
Epochs: 10, LR: 0.01, Alpha: 0.7, Temperature: 4.0


Epoch: [0][0/32] Loss 2.1343 (2.1343) Acc@1 6.250 (6.250)
Epoch: [0][10/32] Loss 2.0966 (2.1871) Acc@1 15.625 (9.943)
Epoch: [0][20/32] Loss 2.1899 (2.1549) Acc@1 12.500 (11.458)
Epoch: [0][30/32] Loss 1.9846 (2.1214) Acc@1 18.750 (13.206)
Epoch [1/10] - Train Loss: 2.1197 (CE: 2.2739, KD: 1.7597) Train Acc: 13.10%
Epoch: [1][0/32] Loss 1.9516 (1.9516) Acc@1 18.750 (18.750)
Epoch: [1][10/32] Loss 1.7984 (1.8674) Acc@1 25.000 (18.750)
Epoch: [1][20/32] Loss 1.8334 (1.8814) Acc@1 18.750 (20.387)
Epoch: [1][30/32] Loss 1.5553 (1.8425) Acc@1 28.125 (22.984)
Epoch [2/10] - Train Loss: 1.8401 (CE: 2.0631, KD: 1.3197) Train Acc: 23.00%
Epoch: [2][0/32] Loss 1.6120 (1.6120) Acc@1 28.125 (28.125)
Epoch: [2][10/32] Loss 1.7918 (1.6660) Acc@1 15.625 (31.250)
Epoch: [2][20/32] Loss 1.7045 (1.6722) Acc@1 37.500 (30.506)
Epoch: [2][30/32] Loss 1.8852 (1.6797) Acc@1 21.875 (29.435)
Epoch [3/10] - Train Loss: 1.6782 (CE: 1.9338, KD: 1.0819) Train Acc: 29.40%
Epoch: [3][0/32] Loss 1.7019 (1.7019) Acc@1

In [3]:
from torchsummary import summary
summary(trained_student_model, input_size=alice_images[0].shape)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 6, 32, 32]             456
            Conv2d-2           [-1, 16, 12, 12]           2,416
            Linear-3                  [-1, 120]          69,240
            Linear-4                   [-1, 84]          10,164
            Linear-5                   [-1, 10]             850
Total params: 83,126
Trainable params: 83,126
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.01
Forward/backward pass size (MB): 0.07
Params size (MB): 0.32
Estimated Total Size (MB): 0.39
----------------------------------------------------------------


### Split Model

Alice also needs to perform split model to split her model $M$ into $A,B,C$. 


In [4]:
from privade.split import split_model
try:
    # Split the model
    model_A, model_B, model_C, split_stats = split_model(
        data_loader=alice_dataloader,
        model=trained_student_model,
    )
    model_A.to(device)
    model_B.to(device)
    model_C.to(device)

    print(f"\nSplit successful!")
    print(f"Optimal boundary layer: {split_stats['optimal_layer']}")
    print(f"Privacy preserved rate: {split_stats['privacy_preserved_rate']:.3f}")
    print(f"Client model (model_B): {len(list(model_B.children()))} layers")
    print(f"Server model (model_C): {len(list(model_C.children()))} layers")
    
except Exception as e:
    print(f"Error during splitting: {e}")
    import traceback
    traceback.print_exc()
    

Identifying candidate layers for B/C boundary...
Found 1 candidate layers: [1]
Starting boundary analysis with DINA attack...
Attack epochs: 50
This may take a while...
Starting boundary layer evaluation...


Evaluating layers:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating layer 1...
    Split layer: 1
    Distillation taps (pre-ReLU conv), ordered near→far: [1, 0]
    Sub-blocks: [[0], [1]]
    Channel path: [16, 16, 6]
    Spatial path: [(28, 28), (28, 28), (32, 32)]
    Loss coefficients: [1.0, 3.0, 6.0]
    Training DINA attack for 50 epochs...


DINA Epoch 1/50: 100%|██████████| 32/32 [00:00<00:00, 193.16it/s, Loss=115.1482]


Epoch 1: Average Loss = 120.6346


...

Epoch 49: Average Loss = 104.1831


DINA Epoch 50/50: 100%|██████████| 32/32 [00:00<00:00, 435.92it/s, Loss=113.0601]


Epoch 50: Average Loss = 104.5179
    Evaluating DINA attack...


Evaluating layers: 100%|██████████| 1/1 [00:03<00:00,  3.95s/it]

  Privacy Preserved Rate: 0.816

BOUNDARY ANALYSIS RESULTS

Layer 1:
  Privacy Preserve Rate: 0.816
  Attack Success: 0.184
  Avg SSIM: 0.222

OPTIMAL BOUNDARY LAYER: 1

Three-Model Split Statistics:
Model A ends at layer: 0
Model B starts at layer: 1, ends at layer: 1
Model C starts at layer: 2
Privacy Rate: 0.816
Attack Success: 0.184
Average SSIM: 0.222

Split successful!
Optimal boundary layer: 1
Privacy preserved rate: 0.816
Client model (model_B): 1 layers
Server model (model_C): 3 layers


In [5]:
#Optionally, add a weight mixer to model A and unmix in model_B
from privade.weight_mixer import weight_mixer

model_A, model_B = weight_mixer(model_A, model_B)

## Part 1: Representative Set selection

## Part 2: Model Inference


## Part 3: Secure Scoring